# P1-1 · Hồi quy tuyến tính và logistic

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import numpy as np
x = np.array([50, 60, 80, 100, 120.]); y = np.array([1.5, 1.8, 2.4, 2.9, 3.6])
w, b = np.polyfit(x, y, 1)      # w ≈ 0.02945, b ≈ 0.025
((w * x + b - y) ** 2).mean()   # MSE ≈ 0.0014

0.0014024390243902551


## Bài tập 3: Logistic regression từ đầu {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

class LogReg:
    def fit(self, X, y, lr=0.1, epochs=2000, lam=1e-3):
        n, d = X.shape
        self.w, self.b = np.zeros(d), 0.0
        for _ in range(epochs):
            p = 1 / (1 + np.exp(-(X @ self.w + self.b)))
            self.w -= lr * (X.T @ (p - y) / n + lam * self.w)
            self.b -= lr * (p - y).mean()
        return self
    def predict_proba(self, X):
        return 1 / (1 + np.exp(-(X @ self.w + self.b)))

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
sc = StandardScaler().fit(Xtr)
Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
ours = ((LogReg().fit(Xtr, ytr).predict_proba(Xte) > 0.5) == yte).mean()
ref = LogisticRegression(max_iter=5000).fit(Xtr, ytr).score(Xte, yte)
print(f"của bạn {ours:.3f} | sklearn {ref:.3f}")

của bạn 0.974 | sklearn 0.982


## Bài tập 4: Định giá căn hộ bằng hồi quy tuyến tính {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
rng = np.random.default_rng(0)
n = 300
area = rng.uniform(35, 130, n)
rooms = np.clip(np.round(area / 30 + rng.normal(0, 0.6, n)), 1, 4)
dist = rng.uniform(1, 20, n)
price = 0.045 * area + 0.15 * rooms - 0.08 * dist + 0.8 + rng.normal(0, 0.3, n)
X = np.column_stack([area, rooms, dist])
# TODO: chia train/test (random_state=0), fit, in hệ số, mae, gia_du_doan

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, price, test_size=0.2, random_state=0)
model = LinearRegression().fit(X_tr, y_tr)
mae = mean_absolute_error(y_te, model.predict(X_te))
gia_du_doan = model.predict([[70, 2, 6]])[0]
for ten, w in zip(["diện tích", "phòng ngủ", "khoảng cách"], model.coef_):
    print(f"{ten:12s}: {w:+.3f} tỷ mỗi đơn vị")
print(f"MAE test = {mae:.3f} tỷ, căn mới ≈ {gia_du_doan:.2f} tỷ")

diện tích   : +0.045 tỷ mỗi đơn vị
phòng ngủ   : +0.130 tỷ mỗi đơn vị
khoảng cách : -0.074 tỷ mỗi đơn vị
MAE test = 0.292 tỷ, căn mới ≈ 3.73 tỷ


## Bài tập 5: Nghiệm đóng hay gradient descent {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n = 300
area = rng.uniform(35, 130, n); rooms = np.clip(np.round(area / 30 + rng.normal(0, 0.6, n)), 1, 4); dist = rng.uniform(1, 20, n)
price = 0.045 * area + 0.15 * rooms - 0.08 * dist + 0.8 + rng.normal(0, 0.3, n)
X = np.column_stack([area, rooms, dist])

def gd(X, y, lr=0.1, steps=500):
    Xb = np.column_stack([np.ones(len(X)), X])          # thêm cột 1 cho bias
    w = np.zeros(Xb.shape[1])
    for _ in range(steps):
        w -= lr * 2 * Xb.T @ (Xb @ w - y) / len(y)
    return w
# TODO: w_dong (nghiệm đóng trên X đã chuẩn hóa), w_goc = gd(X, price), w_chuan = gd(Xs, price)

In [ ]:
Xs = (X - X.mean(0)) / X.std(0)
w_dong = np.linalg.lstsq(np.column_stack([np.ones(n), Xs]), price, rcond=None)[0]
with np.errstate(over="ignore", invalid="ignore"):
    w_goc = gd(X, price)
w_chuan = gd(Xs, price)
print("nghiệm đóng   :", w_dong.round(4))
print("GD chuẩn hóa  :", w_chuan.round(4))
print("GD dữ liệu gốc:", w_goc)

nghiệm đóng   : [ 4.2443  1.2538  0.1639 -0.4216]
GD chuẩn hóa  : [ 4.2443  1.2538  0.1639 -0.4216]
GD dữ liệu gốc: [nan nan nan nan]


## Bài tập 6: Dự đoán khách sắp rời đi {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(1)
n = 800
months = rng.integers(1, 72, n); calls = rng.poisson(1.5, n); fee = rng.uniform(1, 8, n)
logit = -1.0 - 0.05 * months + 0.9 * calls + 0.15 * fee
churn = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
X = np.column_stack([months, calls, fee])
# TODO: chia train/test (random_state=0, stratify), chuẩn hóa, fit, acc, p_khach_moi

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, churn, test_size=0.25, random_state=0, stratify=churn)
sc = StandardScaler().fit(X_tr)
model = LogisticRegression().fit(sc.transform(X_tr), y_tr)
acc = model.score(sc.transform(X_te), y_te)
p_khach_moi = model.predict_proba(sc.transform([[3, 5, 4]]))[0, 1]
print(dict(zip(["tháng dùng", "khiếu nại", "cước"], model.coef_[0].round(2))))
print(f"accuracy test = {acc:.3f}, khách mới: P(rời đi) = {p_khach_moi:.2f}")

{'tháng dùng': -0.98, 'khiếu nại': 1.0, 'cước': 0.18}
accuracy test = 0.755, khách mới: P(rời đi) = 0.97


## Bài tập 7: Regularization khi ít dữ liệu, nhiều đặc trưng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(2)
X = rng.normal(size=(60, 40))
y = 3 * X[:, 0] - 2 * X[:, 1] + 1.5 * X[:, 2] + rng.normal(0, 1, 60)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
# TODO: fit 3 model, r2 (dict tên → R² test), n_zero (số hệ số Lasso bằng 0)

In [ ]:
models = {"Linear": LinearRegression(), "Ridge": Ridge(alpha=1.0), "Lasso": Lasso(alpha=0.1)}
r2 = {}
for ten, m in models.items():
    m.fit(X_tr, y_tr)
    r2[ten] = m.score(X_te, y_te)
    print(f"{ten:6s}: R² train {m.score(X_tr, y_tr):.2f}, R² test {r2[ten]:.2f}")
n_zero = (models["Lasso"].coef_ == 0).sum()
print("Lasso giữ lại các đặc trưng:", np.flatnonzero(models["Lasso"].coef_))

Linear: R² train 1.00, R² test -5.67
Ridge : R² train 0.99, R² test 0.68
Lasso : R² train 0.97, R² test 0.86
Lasso giữ lại các đặc trưng: [ 0  1  2  3  4  6  7  9 12 16 17 22 25 27 28 35 36 37 38]
